In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# --------------------------------------------------
# Function 7 - Week 8 Bayesian Optimisation
# --------------------------------------------------
#
# - maximise raw objective
# - use all observations through Week 7
# - ARD Matern GP
# - automatic GP hyperparameter optimisation
# - larger candidate pools because Function 7 is 6D
# - EI primary
# - UCB used to inspect exploration/exploitation
# - no trust region unless diagnostics justify it

In [2]:
X = np.load("function7/initial_inputs.npy")
Y = np.load("function7/initial_outputs.npy").reshape(-1)

assert len(X) == len(Y)
assert X.shape[1] == 6

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best observed input:")
print(best_x)

print("\nCurrent best observed output:")
print(best_y)

print("\nY range:")
print("min =", np.min(Y))
print("max =", np.max(Y))
print("std =", np.std(Y))

X shape: (37, 6)
Y shape: (37,)

Current best observed input:
[0.093684 0.325604 0.371379 0.230467 0.28146  0.638895]

Current best observed output:
2.66033631734043

Y range:
min = 0.0027014650245082332
max = 2.66033631734043
std = 0.8238679686730516


In [3]:
kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(6) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y)

print("\nFitted kernel:")
print(gp.kernel_)


Fitted kernel:
0.621**2 * Matern(length_scale=[0.354, 2, 2, 0.173, 0.162, 0.394], nu=2.5) + WhiteKernel(noise_level=0.0013)


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [4]:
lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales
relative_sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:")
print(lengthscales)

print("\nNormalised inverse-lengthscale sensitivity:")
print(relative_sensitivity)


ARD lengthscales:
[0.35373441 2.         2.         0.17335097 0.16220715 0.39354353]

Normalised inverse-lengthscale sensitivity:
[0.1544663  0.02732002 0.02732002 0.31519897 0.33685351 0.13884118]


In [5]:
local_scale = np.clip(
    0.25 * lengthscales,
    0.015,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.04,
    0.20
)

print("\nLocal widths:", local_scale)
print("Wide widths:", wide_scale)


Local widths: [0.0884336  0.1        0.1        0.04333774 0.04055179 0.09838588]
Wide widths: [0.17686721 0.2        0.2        0.08667548 0.08110357 0.19677176]


In [6]:
rng = np.random.default_rng(42)

local_candidates = (
    best_x
    + rng.normal(
        0,
        local_scale,
        size=(100000, 6)
    )
)

wide_candidates = (
    best_x
    + rng.normal(
        0,
        wide_scale,
        size=(70000, 6)
    )
)

global_candidates = rng.uniform(
    0,
    1,
    size=(180000, 6)
)

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])

print(
    "Candidates before filtering:",
    len(candidates)
)

Candidates before filtering: 350000


In [7]:
tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print(
    "Candidates after filtering:",
    len(candidates)
)

Candidates after filtering: 349999


In [8]:
mu, sigma = gp.predict(
    candidates,
    return_std=True
)

print("Predictions complete.")

Predictions complete.


In [9]:
def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = mu - best_y - xi

    Z = np.zeros_like(mu)

    valid = sigma > 1e-12

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid] * norm.cdf(Z[valid])
        +
        sigma[valid] * norm.pdf(Z[valid])
    )

    return EI

In [10]:
EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\nPRIMARY EI RESULT")

print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


PRIMARY EI RESULT
candidate = [0.14639511 0.32854476 0.3998783  0.24639822 0.27197027 0.67221551]
mean = 2.684007493398089
std = 0.10758970794809386
EI = 0.05579234199626539


In [11]:
y_scale = np.std(Y)

xi_values = [
    0.0,
    0.01 * y_scale,
    0.05 * y_scale,
    0.10 * y_scale
]

print("\nEI sensitivity check:\n")

for xi in xi_values:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", f"{xi:.6e}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


EI sensitivity check:

xi = 0.000000e+00 
 candidate = [0.14639511 0.32854476 0.3998783  0.24639822 0.27197027 0.67221551] 
 mean = 2.684007 
 std = 0.10759 
 EI = 0.05579234 

xi = 8.238680e-03 
 candidate = [0.14639511 0.32854476 0.3998783  0.24639822 0.27197027 0.67221551] 
 mean = 2.684007 
 std = 0.10759 
 EI = 0.05107913 

xi = 4.119340e-02 
 candidate = [0.14225637 0.28439397 0.50624195 0.25241349 0.26687085 0.68890805] 
 mean = 2.665865 
 std = 0.129109 
 EI = 0.03562749 

xi = 8.238680e-02 
 candidate = [0.16240316 0.17821189 0.41000768 0.25486719 0.26901688 0.69570249] 
 mean = 2.639073 
 std = 0.151122 
 EI = 0.02211358 



In [12]:
print("\nUCB diagnostic:\n")

for beta in [
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = mu + beta * sigma

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


UCB diagnostic:

beta=0.1 
 candidate = [0.1248303  0.25338154 0.40143013 0.24353759 0.27872652 0.6666692 ] 
 mean = 2.690002 
 std = 0.081862 
 UCB = 2.698188 

beta=0.25 
 candidate = [0.14279089 0.22265562 0.46238631 0.2405755  0.27496499 0.67058796] 
 mean = 2.685855 
 std = 0.101166 
 UCB = 2.711146 

beta=0.5 
 candidate = [0.14639511 0.32854476 0.3998783  0.24639822 0.27197027 0.67221551] 
 mean = 2.684007 
 std = 0.10759 
 UCB = 2.737802 

beta=1.0 
 candidate = [0.14225637 0.28439397 0.50624195 0.25241349 0.26687085 0.68890805] 
 mean = 2.665865 
 std = 0.129109 
 UCB = 2.794974 



In [13]:
mean_idx = np.argmax(mu)

print("\nHighest predicted mean:")
print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


Highest predicted mean:
candidate = [0.1248303  0.25338154 0.40143013 0.24353759 0.27872652 0.6666692 ]
mean = 2.690001609032022
std = 0.08186179652967035


In [14]:
# --------------------------------------------------
# Final Function 7 Week 8 selection
# --------------------------------------------------
#
# Week 7 showed that pure exploitation / low-beta UCB
# could overtrust the GP mean in this 6D function.
#
# In Week 8:
# - highest mean and beta=0.1 remain more exploitative
# - beta=0.5 moves modestly toward higher uncertainty
# - primary EI selects exactly the same candidate
#
# This agreement supports beta=0.5 as a better
# exploration-exploitation compromise.

beta = 0.5

UCB = mu + beta * sigma
final_idx = np.argmax(UCB)

week8_candidate = candidates[final_idx]

print("Week 8 Function 7 candidate:")
print(week8_candidate)

print("\nPredicted mean:")
print(mu[final_idx])

print("\nPredicted std:")
print(sigma[final_idx])

print("\nUCB:")
print(UCB[final_idx])

portal = "-".join(
    f"{x:.6f}"
    for x in week8_candidate
)

print("\nPortal format:")
print(portal)

Week 8 Function 7 candidate:
[0.14639511 0.32854476 0.3998783  0.24639822 0.27197027 0.67221551]

Predicted mean:
2.684007493398089

Predicted std:
0.10758970794809386

UCB:
2.737802347372136

Portal format:
0.146395-0.328545-0.399878-0.246398-0.271970-0.672216
